In [1]:
%pdb on

Automatic pdb calling has been turned ON


# Analyzing DHS microdata

Nigeria 2018 - J:\DATA\DHS_PROG_DHS\NGA\2018
India 2015-2016 - J:\DATA\DHS_PROG_DHS\IND\2015_2016

More recent is available, but probably weird due to COVID

## Documentation

DHS 7 recode manual for variable definitions: https://www.dhsprogram.com/pubs/pdf/DHSG4/Recode7_DHS_10Sep2018_DHSG4.pdf.
However, it does not state which variables are in which file.
The `.MAP` files alongside each data file list the variables in it and what they mean.

Hemoglobin variables of interest: 
- HA53: Hemoglobin level in g/dl with 1 implied dcimal
- HA54: Currently pregnant
- HA55 Result of Hemoglobin measuring.
- HA56 Hemoglobin level adjusted by altitude in g/dl with 1 implied decimal. 

Wealth index variables of interest: 
- HV270: The wealth index is a composite measure of a household's cumulative living standard.
The wealth index is calculated using easy-to-collect data on a household’s ownership of
selected assets, such as televisions and bicycles; materials used for housing construction; and
types of water access and sanitation facilities.
Generated with a statistical procedure known as principal components analysis, the wealth
index places individual households on a continuous scale of relative wealth. DHS separates
all interviewed households into five wealth quintiles to compare the influence of wealth on
various population, health and nutrition indicators. The wealth index is presented in the DHS
Final Reports and survey datasets as a background characteristic
- HV271: Wealth index factor score (5 decimals) 

Pregnancy variables of interest: 
- HML18: Pregnancy status from individual questionnaire. For complete woman’s interviews this is
taken from V213. For incomplete woman's interview with anemia testing the pregnancy
status is taken from this section.
BASE: Women with a completed individual questionnaire or when available information
from the anemia testing section.

List of datasets: https://www.dhsprogram.com/data/dataset/Nigeria_Standard-DHS_2018.cfm?flag=1

Instructions on how to calculate everything can be found at: https://www.dhsprogram.com/pubs/pdf/DHSG1/Guide_to_DHS_Statistics_DHS-7_v2.pdf

In [2]:
import pandas as pd, numpy as np

%load_ext autoreload
%autoreload 2

!date

Tue 06 Aug 2024 02:05:07 PM PDT


In [3]:
location = "india"

In [4]:
# Parameters
location = "india"


## Load data, name columns

In [5]:
directory = "/snfs1/DATA/DHS_PROG_DHS/IND/2015_2016/" if location == "india" else "/snfs1/DATA/DHS_PROG_DHS/NGA/2018/"

### WRA

In [6]:
%%time

wra_columns = {
    "v001": "cluster_number",
    "v002": "household_number",
    "v003": "line_number",
    "v005": "weight",
    "v008": "interview_date",
    "v011": "date_of_birth",
    "v190": "wealth_quintile",
}
wra_data = pd.read_stata(
    directory + ("IND_DHS7_2015_2016_WN_IAIR74FL_Y2018M12D06.DTA" if location == "india" else "NGA_DHS7_2018_WN_NGIR7AFL_Y2019M11D05.DTA"),
    columns=wra_columns.keys(),
)

CPU times: user 40.1 s, sys: 5.92 s, total: 46 s
Wall time: 46 s


In [7]:
wra_data = wra_data[wra_columns.keys()].rename(columns=wra_columns)

In [8]:
def recode_wealth_quintile(df):
    return df.map(
        {
            "poorest": "lowest",
            "poorer": "second",
            "middle": "middle",
            "richer": "fourth",
            "richest": "highest",
        }
    )

In [9]:
wra_data["wealth_quintile"] = recode_wealth_quintile(wra_data.wealth_quintile)

In [10]:
wra_data["weight"] = wra_data.weight / 1_000_000

### Births

In [11]:
birth_columns = {
    "v005": "weight",
    "v008": "interview_date",
    "v190": "wealth_quintile",
    "b3": "birth_date",
    "m18": "size_of_child",
    "m19": "birth_weight_kilograms",
    
}
if location == "india":
    birth_columns["s220a"] = "duration_of_pregnancy"
else:
    birth_columns["b20"] = "duration_of_pregnancy"

birth_data = pd.read_stata(
    directory + ("IND_DHS7_2015_2016_BR_IABR74FL_Y2018M12D06.DTA" if location == "india" else "NGA_DHS7_2018_BR_NGBR7AFL_Y2019M11D05.DTA"),
    columns=birth_columns.keys(),
)
birth_data

,v005,v008,v190,b3,m18,m19,s220a
0,191760,1387,middle,1141,NaN,NaN,NaN
1,191760,1387,middle,1117,NaN,NaN,NaN
2,191760,1387,middle,1089,NaN,NaN,NaN
3,191760,1387,richer,1154,NaN,NaN,NaN
4,191760,1387,richer,1129,NaN,NaN,NaN
...,...,...,...,...,...,...,...
1315612,2380715,1385,richer,1346,very large,3250.0,9.0
1315613,2380715,1385,middle,1100,NaN,NaN,NaN
1315614,2380715,1385,middle,1059,NaN,NaN,NaN
1315615,2380715,1385,middle,1027,NaN,NaN,NaN


In [12]:
birth_data = birth_data[birth_columns.keys()].rename(columns=birth_columns)
birth_data["wealth_quintile"] = recode_wealth_quintile(birth_data.wealth_quintile)
birth_data["weight"] = birth_data.weight / 1_000_000
birth_data

,weight,interview_date,wealth_quintile,birth_date,size_of_child,birth_weight_kilograms,duration_of_pregnancy
0,0.191760,1387,middle,1141,NaN,NaN,NaN
1,0.191760,1387,middle,1117,NaN,NaN,NaN
2,0.191760,1387,middle,1089,NaN,NaN,NaN
3,0.191760,1387,fourth,1154,NaN,NaN,NaN
4,0.191760,1387,fourth,1129,NaN,NaN,NaN
...,...,...,...,...,...,...,...
1315612,2.380715,1385,fourth,1346,very large,3250.0,9.0
1315613,2.380715,1385,middle,1100,NaN,NaN,NaN
1315614,2.380715,1385,middle,1059,NaN,NaN,NaN
1315615,2.380715,1385,middle,1027,NaN,NaN,NaN


### Household members

In [13]:
%%time

hhm_columns = {
    "hv001": "cluster_number",
    "hv002": "household_number",
    "hv005": "weight",
    "hv008": "date_of_interview",
    "hvidx": "line_number",
    "hml18": "currently_pregnant",
    "hv105": "age",
    "hv104": "sex",
    "ha0": "index_to_household",
    "ha1": "age_hemoglobin",
    "hv270": "wealth_quintile",
    "ha53": "hemoglobin_raw_adult",
    "ha56": "hemoglobin_adjusted_adult",
    "ha57": "anemia_adult",
    "hc53": "hemoglobin_raw_child",
    "hc56": "hemoglobin_adjusted_child",
    "hc57": "anemia_child", 
}
hhm_data = pd.read_stata(
    directory + ("IND_DHS7_2015_2016_HHM_IAPR74FL_Y2018M12D06.DTA" if location == "india" else "NGA_DHS7_2018_HHM_NGPR7AFL_Y2019M11D05.DTA"),
    columns=hhm_columns.keys(),
)
hhm_data

CPU times: user 10.7 s, sys: 2.94 s, total: 13.6 s
Wall time: 13.6 s


,hv001,hv002,hv005,hv008,hvidx,hml18,hv105,hv104,ha0,ha1,hv270,ha53,ha56,ha57,hc53,hc56,hc57
0,10001,1,191072,1387,1,NaN,51,male,NaN,NaN,middle,NaN,NaN,NaN,NaN,NaN,NaN
1,10001,1,191072,1387,2,"not pregnant, don't know",46,female,2.0,46.0,middle,81.0,81.0,moderate,NaN,NaN,NaN
2,10001,1,191072,1387,3,NaN,22,male,NaN,NaN,middle,NaN,NaN,NaN,NaN,NaN,NaN
3,10001,1,191072,1387,4,"not pregnant, don't know",20,female,4.0,20.0,middle,113.0,113.0,mild,NaN,NaN,NaN
4,10001,9,191072,1387,1,"not pregnant, don't know",40,female,1.0,40.0,richer,116.0,116.0,mild,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2869038,360482,85,2270734,1385,1,NaN,60,male,NaN,NaN,middle,NaN,NaN,NaN,NaN,NaN,NaN
2869039,360482,85,2270734,1385,2,NaN,50,female,NaN,NaN,middle,NaN,NaN,NaN,NaN,NaN,NaN
2869040,360482,96,2270734,1385,1,NaN,66,male,NaN,NaN,middle,NaN,NaN,NaN,NaN,NaN,NaN
2869041,360482,96,2270734,1385,2,"not pregnant, don't know",46,female,2.0,46.0,middle,119.0,119.0,mild,NaN,NaN,NaN


In [14]:
hhm_data = hhm_data[hhm_columns.keys()].rename(columns=hhm_columns)

In [15]:
hhm_data["age"] = hhm_data.age.replace({"95+": 95, "don't know": np.nan}).astype(float)

In [16]:
hhm_data["sex"] = hhm_data.sex.str.title()

In [17]:
# Interesting -- sometimes age is quite off between hemoglobin and base.
hhm_data.loc[(hhm_data.age - hhm_data.age_hemoglobin).sort_values().index]

,cluster_number,household_number,weight,date_of_interview,line_number,currently_pregnant,age,sex,index_to_household,age_hemoglobin,wealth_quintile,hemoglobin_raw_adult,hemoglobin_adjusted_adult,anemia_adult,hemoglobin_raw_child,hemoglobin_adjusted_child,anemia_child
2476814,332896,13,469645,1399,4,"not pregnant, don't know",17.0,Female,4.0,48.0,richest,refused,NaN,NaN,NaN,NaN,NaN
791387,140281,68,892170,1395,4,"not pregnant, don't know",19.0,Female,4.0,49.0,middle,109.0,109.0,mild,NaN,NaN,NaN
845590,140822,66,241894,1394,4,"not pregnant, don't know",20.0,Female,4.0,49.0,poorer,119.0,108.0,mild,NaN,NaN,NaN
1062670,160904,16,2105279,1384,3,"not pregnant, don't know",19.0,Female,3.0,48.0,richest,132.0,132.0,not anemic,NaN,NaN,NaN
2709355,340097,79,339513,1382,3,"not pregnant, don't know",16.0,Female,3.0,44.0,richest,142.0,142.0,not anemic,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2869037,360482,75,2270734,1385,4,NaN,2.0,Male,NaN,NaN,richer,NaN,NaN,NaN,91.0,91.0,moderate
2869038,360482,85,2270734,1385,1,NaN,60.0,Male,NaN,NaN,middle,NaN,NaN,NaN,NaN,NaN,NaN
2869039,360482,85,2270734,1385,2,NaN,50.0,Female,NaN,NaN,middle,NaN,NaN,NaN,NaN,NaN,NaN
2869040,360482,96,2270734,1385,1,NaN,66.0,Male,NaN,NaN,middle,NaN,NaN,NaN,NaN,NaN,NaN


In [18]:
(hhm_data.age - hhm_data.age_hemoglobin).describe()

count    749344.000000
mean         -0.041664
std           0.841376
min         -31.000000
25%           0.000000
50%           0.000000
75%           0.000000
max          31.000000
dtype: float64

In [19]:
hhm_data["pregnant"] = hhm_data.currently_pregnant.map({"not pregnant, don't know": "not_pregnant", "pregnant": "pregnant"})
hhm_data.loc[hhm_data.sex != 'Female', "pregnant"] = "not_pregnant"
hhm_data.loc[(hhm_data.age < 15) | (hhm_data.age >= 50), "pregnant"] = "not_pregnant"
age_bin_edges = [0, 5, 15, 30, 50, 125]
age_group = pd.IntervalIndex(
    pd.cut(hhm_data.age, age_bin_edges, right=False, include_lowest=True)
)
hhm_data["age_start"] = age_group.left
hhm_data["age_end"] = age_group.right

In [20]:
hhm_data["wealth_quintile"] = recode_wealth_quintile(hhm_data.wealth_quintile)
hhm_data["weight"] = hhm_data.weight / 1_000_000

In [21]:
for type in ["child", "adult"]:
    for base_col in ["hemoglobin_raw", "hemoglobin_adjusted"]:
        col = f"{base_col}_{type}"
        hhm_data[col] = (
            hhm_data[col]
            .astype(str)
            .replace(
                {
                    "not tested": np.nan,
                    "not present": np.nan,
                    "refused": np.nan,
                    "other": np.nan,
                }
            )
            .astype(float)
        )

In [22]:
for base_col in ["hemoglobin_raw", "hemoglobin_adjusted", "anemia"]:
    assert (hhm_data.filter(like=base_col).notnull().sum(axis=1) <= 1).all()
    hhm_data[base_col] = np.nan
    # Could use bfill instead of this loop, but it was incredibly slow for me
    for col in hhm_data.filter(like=base_col).columns:
        hhm_data[base_col] = hhm_data[base_col].fillna(hhm_data[col])

In [23]:
assert (hhm_data[(hhm_data.sex == "male") & (hhm_data.age > 5)].hemoglobin_raw.isnull().all())

## Hemoglobin

In [24]:
id_columns = ["cluster_number", "household_number", "line_number"]
other_overlapping_columns = (
    (set(wra_data.columns) & set(hhm_data.columns)) - set(id_columns) - {"weight"}
)
other_overlapping_columns

{'wealth_quintile'}

In [25]:
wra_hhm_joined = wra_data.merge(
    hhm_data.drop(columns=["weight"]),
    on=id_columns,
    suffixes=("_wra", "_hhm"),
    how="left",
)
wra_hhm_joined

,cluster_number,household_number,line_number,weight,interview_date,date_of_birth,wealth_quintile_wra,date_of_interview,currently_pregnant,age,...,anemia_adult,hemoglobin_raw_child,hemoglobin_adjusted_child,anemia_child,pregnant,age_start,age_end,hemoglobin_raw,hemoglobin_adjusted,anemia
0,10001,1,2,0.191760,1387,835,middle,1387,"not pregnant, don't know",46.0,...,moderate,NaN,NaN,NaN,not_pregnant,30.0,50.0,81.0,81.0,moderate
1,10001,1,4,0.191760,1387,1141,middle,1387,"not pregnant, don't know",20.0,...,mild,NaN,NaN,NaN,not_pregnant,15.0,30.0,113.0,113.0,mild
2,10001,9,1,0.191760,1387,903,fourth,1387,"not pregnant, don't know",40.0,...,mild,NaN,NaN,NaN,not_pregnant,30.0,50.0,116.0,116.0,mild
3,10001,9,2,0.191760,1387,1129,fourth,1387,"not pregnant, don't know",21.0,...,not anemic,NaN,NaN,NaN,not_pregnant,15.0,30.0,137.0,137.0,not anemic
4,10001,9,3,0.191760,1387,1154,fourth,1387,"not pregnant, don't know",19.0,...,not anemic,NaN,NaN,NaN,not_pregnant,15.0,30.0,137.0,137.0,not anemic
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
699681,360482,61,5,2.380715,1385,1023,fourth,1385,"not pregnant, don't know",30.0,...,not anemic,NaN,NaN,NaN,not_pregnant,30.0,50.0,125.0,125.0,not anemic
699682,360482,61,7,2.380715,1385,1149,fourth,1385,"not pregnant, don't know",20.0,...,mild,NaN,NaN,NaN,not_pregnant,15.0,30.0,112.0,112.0,mild
699683,360482,62,1,2.380715,1385,817,lowest,1385,"not pregnant, don't know",47.0,...,mild,NaN,NaN,NaN,not_pregnant,30.0,50.0,106.0,103.0,mild
699684,360482,75,2,2.380715,1385,1105,fourth,1385,"not pregnant, don't know",23.0,...,mild,NaN,NaN,NaN,not_pregnant,15.0,30.0,103.0,103.0,mild


In [26]:
for col in other_overlapping_columns:
    assert (wra_hhm_joined[f"{col}_wra"] == wra_hhm_joined[f"{col}_hhm"]).all()
    wra_hhm_joined[col] = wra_hhm_joined[f"{col}_wra"]
    wra_hhm_joined = wra_hhm_joined.drop(columns=[f"{col}_wra", f"{col}_hhm"])

In [27]:
wra_hhm_joined["pregnant"] = wra_hhm_joined.currently_pregnant.map({"not pregnant, don't know": "not_pregnant", "pregnant": "pregnant"})

In [28]:
# https://stackoverflow.com/a/2415343/ with some tweaks
def weighted_avg_and_std(values, weights):
    """
    Return the weighted average and standard deviation.

    They weights are in effect first normalized so that they
    sum to 1 (and so they must not all be 0).

    values, weights -- NumPy ndarrays with the same shape.
    """
    is_nan = np.isnan(values)
    values = values[~is_nan]
    weights = weights[~is_nan]
    average = np.average(values, weights=weights)
    # Fast and numerically precise:
    variance = np.average((values - average) ** 2, weights=weights)
    return pd.Series(
        {
            "mean": average,
            "sd": np.sqrt(variance),
            # https://ngreifer.github.io/WeightIt/reference/ESS.html
            "effective_sample_size": (weights.sum() ** 2) / (weights**2).sum(),
        }
    )

In [29]:
pregnant_with_anemia_status = wra_hhm_joined[(wra_hhm_joined.currently_pregnant == 'pregnant') & wra_hhm_joined.anemia.notnull()]

In [30]:
# Matches table 10.21.1
pregnant_with_anemia_status.weight.sum()

30325.760056

In [31]:
# Within rounding error of table 10.21.1 value
weighted_avg_and_std(
    pregnant_with_anemia_status.anemia == "severe",
    pregnant_with_anemia_status.weight,
)

mean                         0.012959
sd                           0.113096
effective_sample_size    14669.827307
dtype: float64

In [32]:
# Within rounding error of table 10.21.1 value for any anemia
weighted_avg_and_std(
    pregnant_with_anemia_status.anemia.isin(
        ["severe", "moderate", "mild"]
    ),
    pregnant_with_anemia_status.weight,
)

mean                         0.503902
sd                           0.499985
effective_sample_size    14669.827307
dtype: float64

In [33]:
assert (
    (pregnant_with_anemia_status.anemia == "severe")
    == (pregnant_with_anemia_status.hemoglobin_adjusted < 70)
).all()

In [34]:
assert (
    (
        pregnant_with_anemia_status.anemia.isin(
            ["severe", "moderate", "mild"]
        )
    )
    == (pregnant_with_anemia_status.hemoglobin_adjusted < 110)
).all()

In [35]:
adult_hemoglobin_disparities = (
    wra_hhm_joined.groupby(["wealth_quintile", "pregnant"])
    .apply(lambda df: weighted_avg_and_std(df.hemoglobin_adjusted, weights=df.weight))
    .sort_index()
)
adult_hemoglobin_disparities

mean         sd  effective_sample_size
wealth_quintile pregnant                                                  
lowest          not_pregnant  115.022635  16.135241           75400.465764
                pregnant      106.029126  16.025948            4689.833328
second          not_pregnant  116.108207  16.281235           71860.385099
                pregnant      107.167072  16.209312            4047.784565
middle          not_pregnant  116.488978  16.559569           63288.980815
                pregnant      108.455303  15.571656            2576.743299
fourth          not_pregnant  117.310740  16.266386           52818.826999
                pregnant      110.079442  15.559339            2115.822472
highest         not_pregnant  118.402649  15.538198           44871.372481
                pregnant      113.156279  14.858555            2014.207825

In [36]:
adult_hemoglobin_disparities = adult_hemoglobin_disparities.reset_index()
adult_hemoglobin_disparities = pd.concat([
    adult_hemoglobin_disparities.assign(sex="Female", age_start=15, age_end=125),
    # Assumption: males are like non-pregnant WRA
    adult_hemoglobin_disparities[adult_hemoglobin_disparities.pregnant == "not_pregnant"].assign(sex="Male", age_start=15, age_end=125),
])
adult_hemoglobin_disparities = adult_hemoglobin_disparities.set_index(["sex", "age_start", "age_end", "pregnant", "wealth_quintile"])
adult_hemoglobin_disparities

mean         sd  \
sex    age_start age_end pregnant     wealth_quintile                          
Female 15        125     not_pregnant lowest           115.022635  16.135241   
                         pregnant     lowest           106.029126  16.025948   
                         not_pregnant second           116.108207  16.281235   
                         pregnant     second           107.167072  16.209312   
                         not_pregnant middle           116.488978  16.559569   
                         pregnant     middle           108.455303  15.571656   
                         not_pregnant fourth           117.310740  16.266386   
                         pregnant     fourth           110.079442  15.559339   
                         not_pregnant highest          118.402649  15.538198   
                         pregnant     highest          113.156279  14.858555   
Male   15        125     not_pregnant lowest           115.022635  16.135241   
                                      second           116.108207  16.281235   
                                      middle           116.488978  16.559569   
                                      fourth           117.310740  16.266386   
                                      highest          118.402649  15.538198   

                                                       effective_sample_size  
sex    age_start age_end pregnant     wealth_quintile                         
Female 15        125     not_pregnant lowest                    75400.465764  
                         pregnant     lowest                     4689.833328  
                         not_pregnant second                    71860.385099  
                         pregnant     second                     4047.784565  
                         not_pregnant middle                    63288.980815  
                         pregnant     middle                     2576.743299  
                         not_pregnant fourth                    52818.826999  
                         pregnant     fourth                     2115.822472  
                         not_pregnant highest                   44871.372481  
                         pregnant     highest                    2014.207825  
Male   15        125     not_pregnant lowest                    75400.465764  
                                      second                    71860.385099  
                                      middle                    63288.980815  
                                      fourth                    52818.826999  
                                      highest                   44871.372481

In [37]:
child_hemoglobin_disparities = (
    hhm_data[(hhm_data.age <= 5)].assign(age_start=0, age_end=5, pregnant="not_pregnant").groupby(["sex", "age_start", "age_end", "pregnant", "wealth_quintile"])
    .apply(lambda df: weighted_avg_and_std(df.hemoglobin_adjusted, weights=df.weight))
    .sort_index()
)
child_hemoglobin_disparities

mean         sd  \
sex    age_start age_end pregnant     wealth_quintile                          
Female 0         5       not_pregnant lowest           103.655561  14.697789   
                                      second           105.257986  14.743958   
                                      middle           105.397530  15.145306   
                                      fourth           106.765336  15.145327   
                                      highest          107.989017  14.592007   
Male   0         5       not_pregnant lowest           104.012048  14.808158   
                                      second           105.163293  15.071596   
                                      middle           105.230938  15.340462   
                                      fourth           106.561686  15.281601   
                                      highest          107.259027  15.014856   

                                                       effective_sample_size  
sex    age_start age_end pregnant     wealth_quintile                         
Female 0         5       not_pregnant lowest                    17649.049628  
                                      second                    13006.358921  
                                      middle                    10054.395946  
                                      fourth                     7636.224243  
                                      highest                    5219.711658  
Male   0         5       not_pregnant lowest                    18810.904211  
                                      second                    13982.547262  
                                      middle                    10536.774976  
                                      fourth                     7778.376485  
                                      highest                    6278.526164

In [38]:
adolescent_hemoglobin_disparities = pd.DataFrame(columns=child_hemoglobin_disparities.columns, index=child_hemoglobin_disparities.index).droplevel(["age_start", "age_end"])
for group in adolescent_hemoglobin_disparities.index:
    child_values = child_hemoglobin_disparities.droplevel(["age_start", "age_end"]).loc[group]
    adult_values = adult_hemoglobin_disparities.droplevel(["age_start", "age_end"]).loc[group]
    adolescent_hemoglobin_disparities.loc[group] = (child_values * 0.5 + adult_values * 0.5).T


In [39]:
assert adolescent_hemoglobin_disparities.notnull().all().all()
adolescent_hemoglobin_disparities = adolescent_hemoglobin_disparities.reset_index().assign(age_start=5, age_end=15).set_index(list(child_hemoglobin_disparities.index.names))

In [40]:
hemoglobin_disparities = pd.concat([
    child_hemoglobin_disparities,
    adolescent_hemoglobin_disparities,
    adult_hemoglobin_disparities,
])
hemoglobin_disparities

mean         sd  \
sex    age_start age_end pregnant     wealth_quintile                          
Female 0         5       not_pregnant lowest           103.655561  14.697789   
                                      second           105.257986  14.743958   
                                      middle            105.39753  15.145306   
                                      fourth           106.765336  15.145327   
                                      highest          107.989017  14.592007   
Male   0         5       not_pregnant lowest           104.012048  14.808158   
                                      second           105.163293  15.071596   
                                      middle           105.230938  15.340462   
                                      fourth           106.561686  15.281601   
                                      highest          107.259027  15.014856   
Female 5         15      not_pregnant lowest           109.339098  15.416515   
                                      second           110.683097  15.512596   
                                      middle           110.943254  15.852438   
                                      fourth           112.038038  15.705857   
                                      highest          113.195833  15.065103   
Male   5         15      not_pregnant lowest           109.517341  15.471699   
                                      second            110.63575  15.676415   
                                      middle           110.859958  15.950016   
                                      fourth           111.936213  15.773994   
                                      highest          112.830838  15.276527   
Female 15        125     not_pregnant lowest           115.022635  16.135241   
                         pregnant     lowest           106.029126  16.025948   
                         not_pregnant second           116.108207  16.281235   
                         pregnant     second           107.167072  16.209312   
                         not_pregnant middle           116.488978  16.559569   
                         pregnant     middle           108.455303  15.571656   
                         not_pregnant fourth            117.31074  16.266386   
                         pregnant     fourth           110.079442  15.559339   
                         not_pregnant highest          118.402649  15.538198   
                         pregnant     highest          113.156279  14.858555   
Male   15        125     not_pregnant lowest           115.022635  16.135241   
                                      second           116.108207  16.281235   
                                      middle           116.488978  16.559569   
                                      fourth            117.31074  16.266386   
                                      highest          118.402649  15.538198   

                                                      effective_sample_size  
sex    age_start age_end pregnant     wealth_quintile                        
Female 0         5       not_pregnant lowest                   17649.049628  
                                      second                   13006.358921  
                                      middle                   10054.395946  
                                      fourth                    7636.224243  
                                      highest                   5219.711658  
Male   0         5       not_pregnant lowest                   18810.904211  
                                      second                   13982.547262  
                                      middle                   10536.774976  
                                      fourth                    7778.376485  
                                      highest                   6278.526164  
Female 5         15      not_pregnant lowest                   46524.757696  
                                      second                    42433.37201  
        

In [41]:
results_dir = (
    "../results"
)

In [42]:
hemoglobin_disparities["mean"].rename("value").to_csv(
    f"{results_dir}/hemoglobin/mean_disparities/{location}.csv"
)

In [43]:
hemoglobin_disparities["sd"].rename("value").to_csv(
    f"{results_dir}/hemoglobin/sd_disparities/{location}.csv"
)

## Wealth quintile probabilities

Intuitively, you might think these would be equal; but we are looking at subpopulations which can skew.

In [44]:
group_variables = ["sex", "age_start", "age_end", "pregnant"]

In [45]:
wealth_quintile_probabilities = (
    hhm_data.groupby(group_variables + ["wealth_quintile"], observed=True).weight.sum() /
    hhm_data.groupby(group_variables, observed=True).weight.sum()
)
assert np.allclose(wealth_quintile_probabilities.groupby(group_variables, observed=True).sum(), 1.0)
wealth_quintile_probabilities

sex     age_start  age_end  pregnant      wealth_quintile
Female  0.0        5.0      not_pregnant  lowest             0.251317
                                          second             0.220571
                                          middle             0.195838
                                          fourth             0.184456
                                          highest            0.147818
        5.0        15.0     not_pregnant  lowest             0.269038
                                          second             0.219459
                                          middle             0.193766
                                          fourth             0.172569
                                          highest            0.145167
        15.0       30.0     not_pregnant  lowest             0.179314
                                          second             0.203535
                                          middle             0.211692
                                

In [46]:
wealth_quintile_probabilities = wealth_quintile_probabilities.unstack()
wealth_quintile_probabilities

wealth_quintile                          lowest    second    middle    fourth  \
sex    age_start age_end pregnant                                               
Female 0.0       5.0     not_pregnant  0.251317  0.220571  0.195838  0.184456   
       5.0       15.0    not_pregnant  0.269038  0.219459  0.193766  0.172569   
       15.0      30.0    not_pregnant  0.179314  0.203535  0.211692  0.209712   
                         pregnant      0.221262  0.222487  0.214771  0.182324   
       30.0      50.0    not_pregnant  0.174181  0.188673  0.199713  0.213463   
                         pregnant      0.314491  0.182529  0.131633  0.167147   
       50.0      125.0   not_pregnant  0.188708  0.187540  0.191253  0.199636   
Male   0.0       5.0     not_pregnant  0.242600  0.215610  0.200229  0.183923   
       5.0       15.0    not_pregnant  0.261604  0.217675  0.190718  0.176545   
       15.0      30.0    not_pregnant  0.169172  0.205794  0.214029  0.208330   
       30.0      50.0    not_pregnant  0.167563  0.183939  0.203419  0.219906   
       50.0      125.0   not_pregnant  0.176575  0.184392  0.190461  0.200876   

wealth_quintile                         highest  
sex    age_start age_end pregnant                
Female 0.0       5.0     not_pregnant  0.147818  
       5.0       15.0    not_pregnant  0.145167  
       15.0      30.0    not_pregnant  0.195747  
                         pregnant      0.159154  
       30.0      50.0    not_pregnant  0.223970  
                         pregnant      0.204200  
       50.0      125.0   not_pregnant  0.232863  
Male   0.0       5.0     not_pregnant  0.157639  
       5.0       15.0    not_pregnant  0.153458  
       15.0      30.0    not_pregnant  0.202675  
       30.0      50.0    not_pregnant  0.225173  
       50.0      125.0   not_pregnant  0.247696

In [47]:
wealth_quintile_probabilities.to_csv(
    f"{results_dir}/wealth_quintile_probabilities/{location}.csv",
)

## LBWSG

### Birth weight

In [48]:
birth_data["birth_weight_kilograms"] = birth_data.birth_weight_kilograms.replace(
    {"not weighed at birth": np.nan, "don't know": np.nan}
).astype(float)

In [49]:
weighted_avg_and_std(birth_data.birth_weight_kilograms, birth_data.weight)

mean                      2799.392719
sd                         604.338123
effective_sample_size    86484.148885
dtype: float64

In [50]:
birth_weight_disparities = birth_data.groupby("wealth_quintile").apply(
    lambda df: weighted_avg_and_std(df.birth_weight_kilograms, df.weight)
)
birth_weight_disparities

,mean,sd,effective_sample_size
wealth_quintile,,,
lowest,2767.050071,647.661212,23969.631404
second,2769.538275,608.623042,22126.156274
middle,2793.091464,594.535582,18888.148937
fourth,2811.885429,596.065203,14336.624949
highest,2861.478555,567.034576,12220.438182


In [51]:
birth_weight_disparities = (
    birth_weight_disparities["mean"].rename("value").reset_index()
)
birth_weight_disparities

,wealth_quintile,value
0,lowest,2767.050071
1,second,2769.538275
2,middle,2793.091464
3,fourth,2811.885429
4,highest,2861.478555


In [52]:
birth_weight_disparities.to_csv(
    f"{results_dir}/birth_weight_disparities/{location}.csv", index=False
)

### Short gestation

All we have here is a self-reported duration of pregnancy.

In [53]:
# Basically no difference in mean
birth_data.groupby("wealth_quintile").apply(
    lambda df: weighted_avg_and_std(df.duration_of_pregnancy, df.weight)
)

,mean,sd,effective_sample_size
wealth_quintile,,,
lowest,8.984584,0.506989,46757.969967
second,9.015559,0.518205,34263.878995
middle,9.050525,0.525370,25463.538749
fourth,9.051636,0.531054,18054.353166
highest,9.042706,0.535222,13759.568526


In [54]:
birth_data["short_gestation"] = np.where(
    birth_data.duration_of_pregnancy.isnull(),
    np.nan,
    birth_data.duration_of_pregnancy < 9.0,
)

In [55]:
weighted_avg_and_std(birth_data.short_gestation, birth_data.weight)

mean                          0.073412
sd                            0.260812
effective_sample_size    129817.319675
dtype: float64

In [56]:
birth_data.groupby("wealth_quintile").apply(
    lambda df: weighted_avg_and_std(df.short_gestation, df.weight)
)

,mean,sd,effective_sample_size
wealth_quintile,,,
lowest,0.078308,0.268655,46757.969967
second,0.070505,0.255996,34263.878995
middle,0.067307,0.250553,25463.538749
fourth,0.072144,0.258726,18054.353166
highest,0.079073,0.269852,13759.568526


The trends in short gestation don't make intuitive sense (?), so we do not plan to use them in the sim.